# Neural Networks and LLMs:

Today we'll work from Traditional ML to Neural Networks to Large Language Models!!

## Imports and Loading Dataset:

In [1]:
# Imports:
import os

import pylab as p
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split

In [2]:
# Logging in to Hugging Face:
load_dotenv(override= True)
hf_token = os.getenv('HF_TOKEN')
login(token= hf_token, add_to_git_credential= True)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
# Loading Dataset (Full Version):
username = 'ed-donner'
dataset = f"{username}/items_full"
train, val, test = Item.from_hub(dataset_name= dataset)
print(f"Loaded {len(train)} training items, {len(val)} validation items, {len(test)} test items")

Loaded 800000 training items, 10000 validation items, 10000 test items


## Simple Deep Neural Network:

In [5]:
# Documents (item Summary) and Prices:
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [6]:
# Using the HashingVectorizer for a Bag of Words Model
# Using binary=True with the CountVectorizer makes "one-hot vectors"

np.random.seed(42)
vectorizer = HashingVectorizer(n_features= 5000, stop_words= 'english', binary= True)
X = vectorizer.fit_transform(documents)

### Defining The Neural Network using pytorch:

In [7]:
class NeuralNetwork(nn.Module):

    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()

        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

### Converting Data to Pytorch tensors and Batching:

In [8]:
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Train Test Split:
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size= 0.1, random_state= 42)

# Data Loader:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Initializing The Model:
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

In [9]:
# Number of Parameters in the Neural Network:

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Number of trainable parameters: {trainable_params:,}')

Number of trainable parameters: 669,249
